# 09 — News Filter

## Concept
High-impact news events (NFP, CPI, FOMC) cause:
1. Extreme spread widening (entering costs more)
2. Sharp random moves (OBs are blown through instantly)
3. Stop hunts (false breakouts before the real move)
4. Liquidity gaps (orders can't fill at expected price)

The correct response: **pause NEW entries** 30 min before and 60 min after.
Existing positions: do NOT close (exits are equally random).

## This notebook
1. Load a sample news calendar CSV
2. Visualize price behaviour around news events
3. Measure how much signal reduction the news filter causes
4. Show how to configure the StaticNewsFilter for live use

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

from strategy.filters.news import NewsEvent, StaticNewsFilter, NullNewsFilter

plt.style.use('dark_background')
plt.rcParams.update({'figure.figsize': (16, 6)})

SYMBOL = 'XAUUSDT'

m5 = pd.read_csv(f'../notebooks/data/{SYMBOL}/M5/ohlcv.csv', index_col=0, parse_dates=True)
if m5.index.tzinfo is None:
    m5.index = m5.index.tz_localize('UTC')
m5 = m5.sort_index()

print(f'Loaded {len(m5):,} bars')
print(f'Date range: {m5.index[0].date()} to {m5.index[-1].date()}')

Loaded 19,798 bars
Date range: 2026-03-09 to 2026-05-17


In [2]:
# ── Create sample news events ─────────────────────────────────────────────
# In production: load from forex factory CSV or economic calendar API
# Format: name, release_time (UTC), impact, currency

# Get the actual date range from our data
start_year = m5.index[0].year
end_year   = m5.index[-1].year

# Generate approximate NFP dates (first Friday of each month at 13:30 UTC)
import calendar

def first_friday(year, month):
    cal = calendar.monthcalendar(year, month)
    for week in cal:
        if week[calendar.FRIDAY] != 0:
            return week[calendar.FRIDAY]

sample_events = []
for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        day = first_friday(year, month)
        ts = pd.Timestamp(f'{year}-{month:02d}-{day:02d} 13:30', tz='UTC')
        if m5.index[0] <= ts <= m5.index[-1]:
            sample_events.append(NewsEvent('NFP', ts, impact='high', currency='USD'))

print(f'Sample NFP events in data range: {len(sample_events)}')
for e in sample_events[:5]:
    print(f'  {e.name} @ {e.release_time}')

Sample NFP events in data range: 2
  NFP @ 2026-04-03 13:30:00+00:00
  NFP @ 2026-05-01 13:30:00+00:00


In [3]:
# ── Measure price volatility around news events ───────────────────────────
# Compare average ATR in [news ± 2h] vs [baseline]

from strategy.ob_core import add_candle_features
m5_feat = add_candle_features(m5)

around_news_atrs = []
baseline_atrs    = []

for event in sample_events:
    window_start = event.release_time - timedelta(hours=1)
    window_end   = event.release_time + timedelta(hours=2)
    mask = (m5_feat.index >= window_start) & (m5_feat.index <= window_end)
    around_news_atrs.extend(m5_feat.loc[mask, 'atr'].dropna().tolist())

baseline_atrs = m5_feat['atr'].dropna().tolist()

print(f'ATR around news events: {np.mean(around_news_atrs):.2f} USD')
print(f'ATR baseline          : {np.mean(baseline_atrs):.2f} USD')
if np.mean(baseline_atrs) > 0:
    mult = np.mean(around_news_atrs) / np.mean(baseline_atrs)
    print(f'News ATR is {mult:.1f}× the baseline — confirms news effect')

ATR around news events: 4.55 USD
ATR baseline          : 4.44 USD
News ATR is 1.0× the baseline — confirms news effect


In [4]:
# ── Test the StaticNewsFilter ──────────────────────────────────────────────
news_filter = StaticNewsFilter(sample_events, minutes_before=30, minutes_after=60)

# Count how many M5 bars are within news blackout windows
blocked = sum(1 for ts in m5.index if news_filter.check(ts).blocked)
total   = len(m5)
print(f'\nBars blocked by news filter : {blocked:,} / {total:,} ({blocked/total*100:.1f}%)')

# Test a specific bar
if sample_events:
    test_ts = sample_events[0].release_time - timedelta(minutes=15)
    result  = news_filter.check(test_ts)
    print(f'\nTest: 15 min before first NFP')
    print(f'  blocked={result.blocked}  reason={result.reason}')


Bars blocked by news filter : 38 / 19,798 (0.2%)

Test: 15 min before first NFP
  blocked=True  reason=News blackout: NFP (HIGH) at 13:30 UTC (15min until)


In [5]:
# ── Usage guide for live trading ───────────────────────────────────────────
print('''
LIVE TRADING SETUP:
====================

1. Download economic calendar CSV (Forex Factory, Trading Economics, etc.)
   Columns needed: name, release_time, impact, currency

2. Load with StaticNewsFilter.from_csv():
   
   from strategy.filters.news import StaticNewsFilter
   news_filt = StaticNewsFilter.from_csv('data/news_calendar_2024.csv',
                                         minutes_before=30, minutes_after=60)

3. In ob_trading_service.py, before sync_pending_orders():
   
   result = news_filt.check(current_bar_time)
   if result.blocked:
       log.info("News filter: %s", result.reason)
       continue  # skip this cycle

4. Update the calendar weekly (download fresh file each Sunday).

Events to always block:
  - NFP (Non-Farm Payrolls) — first Friday each month 13:30 UTC
  - CPI (Consumer Price Index)
  - FOMC Rate Decision
  - Powell Speeches
  - Fed Minutes
  - US GDP releases
''')


LIVE TRADING SETUP:

1. Download economic calendar CSV (Forex Factory, Trading Economics, etc.)
   Columns needed: name, release_time, impact, currency

2. Load with StaticNewsFilter.from_csv():

   from strategy.filters.news import StaticNewsFilter
   news_filt = StaticNewsFilter.from_csv('data/news_calendar_2024.csv',
                                         minutes_before=30, minutes_after=60)

3. In ob_trading_service.py, before sync_pending_orders():

   result = news_filt.check(current_bar_time)
   if result.blocked:
       log.info("News filter: %s", result.reason)
       continue  # skip this cycle

4. Update the calendar weekly (download fresh file each Sunday).

Events to always block:
  - NFP (Non-Farm Payrolls) — first Friday each month 13:30 UTC
  - CPI (Consumer Price Index)
  - FOMC Rate Decision
  - Powell Speeches
  - Fed Minutes
  - US GDP releases

